## Trainning The Evaluation Function Weights


In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential
from keras.layers import Dense

### About Data
- data is taken from stockfish static evaluation (depth=0)
- our current evaluation function is static evaluation.
- we havent added check/checkmate evaluation so check positions are removed from dataset to reduce huge fluctuations in Error.

### Representing Data in 2D grid format

In [5]:

EMPTY = 0
# Piece Representations
BLACK_PAWN = -1
BLACK_ROOK = -2
BLACK_KNIGHT = -3
BLACK_BISHOP = -4
BLACK_QUEEN = -5
BLACK_KING = -6

WHITE_PAWN = 1
WHITE_ROOK = 2
WHITE_KNIGHT = 3
WHITE_BISHOP = 4
WHITE_QUEEN = 5
WHITE_KING = 6


def fromFEN(fen):
    # Creates an Empty Board of 8x8
    board = []
    for i in range(8):
        row=[]
        for j in range(8):
            row.append(EMPTY)
        board.append(row)

    row = 0
    col = 0

    for c in fen:
        # if c reaches an empty character then board representation ends
        if c==' ':
            break
        # if / is encountered move to next row
        if c == '/':
            row += 1
            col = 0
        # if a digit is encountered then skip that many consecutive squares
        elif c.isdigit():
            col += int(c)
        # if character is found then place it on current row and column
        else:
            if c == 'P':
                board[row][col] = WHITE_PAWN
            elif c == 'R':
                board[row][col] = WHITE_ROOK
            elif c == 'N':
                board[row][col] = WHITE_KNIGHT
            elif c == 'B':
                board[row][col] = WHITE_BISHOP
            elif c == 'Q':
                board[row][col] = WHITE_QUEEN
            elif c == 'K':
                board[row][col] = WHITE_KING

            elif c == 'p':
                board[row][col] = BLACK_PAWN
            elif c == 'r':
                board[row][col] = BLACK_ROOK
            elif c == 'n':
                board[row][col] = BLACK_KNIGHT
            elif c == 'b':
                board[row][col] = BLACK_BISHOP
            elif c == 'q':
                board[row][col] = BLACK_QUEEN
            elif c == 'k':
                board[row][col] = BLACK_KING

            col += 1

    return board


data=pd.read_csv("data_train.csv")
data_dev=pd.read_csv("data_dev.csv")
data_test=pd.read_csv("data_test.csv")
data=data.dropna()
data_dev=data_dev.dropna()
data_test=data_test.dropna()
tp=fromFEN(data["fen"][0])
tp

[[0, 0, 0, 2, 0, 0, 0, 0],
 [0, 0, 0, -6, 0, -1, 0, -1],
 [0, 0, 0, 0, 0, 0, -1, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0, 1, 1, 1],
 [0, 0, 0, 0, 0, 0, 6, 0]]

### getting features from current board representation

In [6]:
import subprocess

X=[]

def getFeatures(dataFrame):
    boards = []

    for data_str in dataFrame["fen"]:
        arr = fromFEN(data_str)

        for row in arr:
            boards.append(" ".join(map(str, row)))

    inp = f"{len(dataFrame)}\n" + "\n".join(boards) + "\n"

    result = subprocess.run(
        ["../feature"],
        input=inp,
        text=True,
        capture_output=True
    )

    return [
        list(map(float, line.split()))
        for line in result.stdout.strip().splitlines()
    ]

X=getFeatures(data)

X_train=pd.DataFrame(
    X,
    columns=["material","mobility","pawns","pressure","threat","rook","check","KingSafety","kingPosition"]
)

X=getFeatures(data_dev)
X_dev=pd.DataFrame(
    X,
    columns=["material","mobility","pawns","pressure","threat","rook","check","KingSafety","kingPosition"]
)

X=getFeatures(data_test)
X_test=pd.DataFrame(
    X,
    columns=["material","mobility","pawns","pressure","threat","rook","check","KingSafety","kingPosition"]
)
X_train.head()

,material,mobility,pawns,pressure,threat,rook,check,KingSafety,kingPosition
0,868.0,16.0,50.0,0.0,0.0,15.0,1.0,90.0,-40.0
1,-10.0,-3.0,0.0,10.0,0.0,0.0,0.0,-540.0,0.0
2,8.0,12.0,75.0,0.0,-30.0,0.0,0.0,-60.0,0.0
3,138.0,30.0,-15.0,30.0,0.0,0.0,0.0,-520.0,-10.0
4,-348.0,-56.0,15.0,20.0,0.0,-15.0,0.0,-150.0,0.0


In [7]:
X_test

,material,mobility,pawns,pressure,threat,rook,check,KingSafety,kingPosition
0,827.0,58.0,0.0,-10.0,0.0,0.0,0.0,-520.0,0.0
1,-500.0,-19.0,-15.0,0.0,0.0,0.0,0.0,-490.0,20.0
2,-140.0,8.0,-65.0,0.0,0.0,-15.0,0.0,-300.0,30.0
3,-87.0,23.0,-35.0,-10.0,0.0,0.0,0.0,-470.0,0.0
4,-165.0,-22.0,-35.0,60.0,80.0,-15.0,0.0,-590.0,0.0
...,...,...,...,...,...,...,...,...,...
4927,-125.0,30.0,0.0,-30.0,0.0,0.0,0.0,-370.0,0.0
4928,0.0,40.0,0.0,0.0,0.0,0.0,0.0,-20.0,0.0
4929,-433.0,18.0,0.0,0.0,-10.0,0.0,0.0,-950.0,0.0
4930,15.0,28.0,0.0,0.0,0.0,0.0,0.0,-600.0,0.0


In [20]:
scale=StandardScaler()
scale.fit(X_train)

X_train_scaled=scale.transform(X_train)
X_dev_scaled=scale.transform(X_dev)
X_test_scaled=scale.transform(X_test)


Y_train=data[["score"]]
Y_dev=data_dev[["score"]]
Y_test=data_test[["score"]]

model = Sequential([
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(64,activation='relu'),
    Dense(32,activation='relu'),
    Dense(20,activation='relu'),
    Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

import numpy as np
import os

# ==========================================
# TRAIN
# ==========================================

model.fit(X_train_scaled, Y_train)


# ==========================================
# VERIFY SCALER
# ==========================================

print("SCALER MEAN:")
print(scale.mean_)

print("\nSCALER SCALE:")
print(scale.scale_)


# ==========================================
# EXPORT C++ HEADER
# ==========================================

output_file = r"C:\Code\Chess\Neural_Network\nn_weights.h"

with open(output_file, "w") as f:

    f.write("#ifndef NN_WEIGHTS_H\n")
    f.write("#define NN_WEIGHTS_H\n\n")

    # --------------------------------------
    # StandardScaler
    # --------------------------------------

    f.write("const double SCALER_MEAN[9] = {\n")

    for value in scale.mean_:
        f.write(f"    {value:.17g},\n")

    f.write("};\n\n")

    f.write("const double SCALER_SCALE[9] = {\n")

    for value in scale.scale_:
        f.write(f"    {value:.17g},\n")

    f.write("};\n\n")


    # --------------------------------------
    # Neural network
    # --------------------------------------

    for layer_num, layer in enumerate(model.layers, start=1):

        weights, bias = layer.get_weights()

        print(
            f"Layer {layer_num}: "
            f"weights={weights.shape}, "
            f"bias={bias.shape}"
        )

        # IMPORTANT:
        # Keep Keras's original layout.
        #
        # W.shape = (input_size, output_size)

        rows, cols = weights.shape

        f.write(
            f"const double W{layer_num}"
            f"[{rows}][{cols}] = {{\n"
        )

        for row in weights:

            f.write("    {")

            for value in row:
                f.write(f"{value:.17g}, ")

            f.write("},\n")

        f.write("};\n\n")


        # ----------------------------------
        # Bias
        # ----------------------------------

        f.write(
            f"const double B{layer_num}"
            f"[{len(bias)}] = {{\n"
        )

        for value in bias:
            f.write(f"    {value:.17g},\n")

        f.write("};\n\n")


    f.write("#endif\n")


print("\n================================")
print("nn_weights.h CREATED")
print("================================")
print("Path:", output_file)
print("Exists:", os.path.exists(output_file))
print("Size:", os.path.getsize(output_file), "bytes")

Y_pred_dev=model.predict(X_dev_scaled)
Y_pred_test=model.predict(X_test_scaled)
Y_pred_train=model.predict(X_train_scaled)

err_train=root_mean_squared_error(Y_train,Y_pred_train)
err_test= root_mean_squared_error(Y_test,Y_pred_test)
err_dev = root_mean_squared_error(Y_dev,Y_pred_dev)


print("Dev set Error: ",err_dev)
print("Train set Error: ",err_train)
print("Test set Error:",err_test)

6168/6168 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - loss: 35609.1406 - mae: 132.4605
SCALER MEAN:
[ 2.36813919e+00  3.83797012e+00  3.26061073e-01 -1.93696136e-01
  1.57571274e-02  5.10784259e-01  2.00637378e-03 -3.32734900e+02
  7.29590467e-03]

SCALER SCALE:
[4.67087528e+02 3.32560529e+01 3.72743113e+01 2.29678857e+01
 2.36425017e+01 1.07882542e+01 2.72417947e-01 2.89463779e+02
 1.90042473e+01]
Layer 1: weights=(9, 128), bias=(128,)
Layer 2: weights=(128, 64), bias=(64,)
Layer 3: weights=(64, 64), bias=(64,)
Layer 4: weights=(64, 32), bias=(32,)
Layer 5: weights=(32, 20), bias=(20,)
Layer 6: weights=(20, 1), bias=(1,)

nn_weights.h CREATED
Path: C:\Code\Chess\Neural_Network\nn_weights.h
Exists: True
Size: 364161 bytes
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 986us/step
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 762us/step
6168/6168 ━━━━━━━━━━━━━━━━━━━━ 4s 658us/step
Dev set Error:  175.85403442382812
Train set Error:  182.64871215820312
Test set Error: 180.63629150390625


In [9]:
import numpy as np

corr_dev = np.corrcoef(
    Y_dev.values.flatten(),
    Y_pred_dev.flatten()
)[0,1]

corr_train = np.corrcoef(
    Y_train.values.flatten(),
    Y_pred_train.flatten()
)[0,1]

corr_test = np.corrcoef(
    Y_test.values.flatten(),
    Y_pred_test.flatten()
)[0,1]

print("Correlation of Dev set:",corr_dev)
print("Correlation of Train set:",corr_train)
print("Correlation of Test set:",corr_test)

Correlation of Dev set: 0.849632461634263
Correlation of Train set: 0.8521429175940659
Correlation of Test set: 0.8577553139826954


In [ ]:
import numpy as np

# ==========================================
# EXPORT STANDARD SCALER
# ==========================================

with open("nn_weights.h", "w") as f:

    f.write("#ifndef NN_WEIGHTS_H\n")
    f.write("#define NN_WEIGHTS_H\n\n")

    # --------------------------
    # StandardScaler
    # --------------------------

    f.write("const double SCALER_MEAN[9] = {\n")

    for x in scale.mean_:
        f.write(f"    {x:.17g},\n")

    f.write("};\n\n")

    f.write("const double SCALER_SCALE[9] = {\n")

    for x in scale.scale_:
        f.write(f"    {x:.17g},\n")

    f.write("};\n\n")

    # ==========================================
    # NEURAL NETWORK WEIGHTS
    # ==========================================

    for layer_num, layer in enumerate(model.layers, start=1):

        weights, bias = layer.get_weights()

        # --------------------------------------
        # Weight matrix
        # --------------------------------------

        rows, cols = weights.shape

        f.write(
            f"const double W{layer_num}[{rows}][{cols}] = {{\n"
        )

        for row in weights:
            f.write("    {")

            for value in row:
                f.write(f"{value:.17g}, ")

            f.write("},\n")

        f.write("};\n\n")

        # --------------------------------------
        # Bias
        # --------------------------------------

        f.write(
            f"const double B{layer_num}[{len(bias)}] = {{\n"
        )

        for value in bias:
            f.write(f"    {value:.17g},\n")

        f.write("};\n\n")

    f.write("#endif\n")

print("Weights exported to nn_weights.h")

Created: C:\Code\Chess\Neural_Network\nn_weights.h


In [21]:
x = np.array([827., 58., 0., -10., 0., 0., 0., -520., 0.])

x_scaled = scale.transform(x.reshape(1, -1))[0]

W, B = model.layers[0].get_weights()

z = np.dot(x_scaled, W) + B
a = np.maximum(0, z)

print("Python scaled:")
print(x_scaled)

print("\nPython first 10 after ReLU:")
print(a[:10])

print("\nPython prediction:")
print(model.predict(x_scaled.reshape(1, -1), verbose=0)[0][0])

Python scaled:
[ 1.76547609e+00  1.62863675e+00 -8.74760825e-03 -4.26957186e-01
 -6.66474624e-04 -4.73463315e-02 -7.36505727e-03 -6.46937937e-01
 -3.83909163e-04]

Python first 10 after ReLU:
[0.61243719 0.90958266 0.10806123 0.78967289 0.         0.05880455
 0.         0.         0.55020674 0.47396425]

Python prediction:
526.79974


C:\Users\athar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
